# 06a — Faster R-CNN + GrabCut E2E Evaluation

**Objective**:

Run end-to-end (E2E) evaluation on the test split (**1,733 test images / 6,398 pairs**) for **Faster R-CNN** (`fasterrcnn_mobilenet_v3_large_320_fpn`) coupled with **GrabCut** segmentation, reproducing the classical multi-stage baseline from the original ECUSTFD paper.

### Key Pipeline Components:
1. **Bounding Box Detection**: Faster R-CNN detects food instances and the reference coin.
2. **GrabCut Segmentation**: Crops detected bounding boxes and applies GrabCut to segment foreground food masks.
3. **Coin Calibration**: Computes physical scale (cm/px) from the reference coin ($2.5\text{ cm}$ diameter) with relaxed coin-gating.
4. **View Pairing**: Matches corresponding top-view and side-view images.
5. **3D Volume & Calorie Estimation**: Computes volume using geometric shape formulas and applies $\beta$-calibration to estimate mass and calories.
6. **Artifacts**: Exports evaluation reports (`summary.txt`, `report_test_*.json`, `samples_test_*.csv`, `betas_train_*.json`, `speed_per_image.json`).


In [1]:
"""Cell 1 -- Setup Dual Logging (Console + File Log)."""
import sys
from pathlib import Path

PROJECT_ROOT = Path("E:/AI_Research/dlt8").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from src.faster_rcnn.two_stage_helpers import setup_eval_logging

log, LOG_PATH, RUN_DIR, RUN_TS = setup_eval_logging("06a_faster_rcnn_eval", PROJECT_ROOT)
log.info("Logger initialized.")


2026-09-03 13:13:03 [INFO] ======================================================================
2026-09-03 13:13:03 [INFO] === 06A_FASTER_RCNN_EVAL -- EVALUATION SESSION STARTED ===
2026-09-03 13:13:03 [INFO] ======================================================================
2026-09-03 13:13:03 [INFO] PROJECT_ROOT : E:\AI_Research\dlt8
2026-09-03 13:13:03 [INFO] LOG_PATH     : E:\AI_Research\dlt8\outputs\logs\06a_faster_rcnn_eval_20260903-131303.log
2026-09-03 13:13:03 [INFO] RUN_DIR      : E:\AI_Research\dlt8\outputs\predictions\06a_faster_rcnn_eval_20260903-131303
2026-09-03 13:13:03 [INFO] TIMESTAMP    : 20260903-131303
2026-09-03 13:13:03 [INFO] Logger initialized.


In [2]:
"""Cell 2 -- Imports and dependencies."""
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import torch

from src.constants import FOOD_CLASSES, SHAPE_MODELS
from src.faster_rcnn.inference import _IDX_TO_NAME, load_model, predict_one
from src.faster_rcnn.two_stage_helpers import (
    sanity_check_label_mapping,
    fix_ground_truth_aliasing,
    run_smoke_test,
    compute_speed_report,
    format_and_print_report,
)
from src.e2e_pipeline.dataset_split import (
    load_split,
    resolve_image_paths,
    group_top_side,
    make_pairs,
)

log.info("Imports OK.")
log.info("  FOOD_CLASSES (%d): %s", len(FOOD_CLASSES), ", ".join(FOOD_CLASSES))


2026-09-03 13:13:03 [INFO] Imports OK.
2026-09-03 13:13:03 [INFO]   FOOD_CLASSES (19): apple, banana, bread, bun, doughnut, egg, fried_dough_twist, grape, lemon, litchi, mango, mooncake, orange, peach, pear, plum, qiwi, sachima, tomato


In [3]:
"""Cell 3 -- Config and hyperparameters."""
MODEL_VARIANT = "new"
MASK_BACKEND = "grabcut"
CONF = 0.1   # frozen from sweep 20260902-234626 (test_tune): min MAE duoi rang buoc giu nguyen coverage ~100%
IOU_THRESH = 0.50
GRABCUT_ITERS = 5
SPLIT = "test_final"  # held-out final half
DEVICE = "0" if torch.cuda.is_available() else "cpu"

# Set MAX_PAIRS = 5 for fast validation test, or None for full test (1733 images)
MAX_PAIRS = None

MODEL_WEIGHTS = PROJECT_ROOT / "models" / "faster_rcnn_new_best.pt"
IMAGES_DIR = PROJECT_ROOT / "data" / "raw" / "ECUSTFD" / "JPEGImages"
IMAGESETS_DIR = PROJECT_ROOT / "data" / "raw" / "ECUSTFD" / "ImageSets" / "Main"

TAG = f"{SPLIT}_conf{int(CONF*100)}_beta"
CSV_PATH = RUN_DIR / f"samples_{TAG}.csv"
JSON_PATH = RUN_DIR / f"report_{TAG}.json"
BETA_JSON = RUN_DIR / f"betas_train_conf{int(CONF*100)}.json"

log.info("Configuration:")
log.info("  Model Weights : %s (exists=%s)", MODEL_WEIGHTS, MODEL_WEIGHTS.exists())
log.info("  Mask Backend  : %s (iters=%d)", MASK_BACKEND, GRABCUT_ITERS)
log.info("  Confidence    : %.2f | IoU: %.2f", CONF, IOU_THRESH)
log.info("  Max Pairs     : %s", MAX_PAIRS if MAX_PAIRS else "All")


2026-09-03 13:13:03 [INFO] Configuration:
2026-09-03 13:13:03 [INFO]   Model Weights : E:\AI_Research\dlt8\models\faster_rcnn_new_best.pt (exists=True)
2026-09-03 13:13:03 [INFO]   Mask Backend  : grabcut (iters=5)
2026-09-03 13:13:03 [INFO]   Confidence    : 0.10 | IoU: 0.50
2026-09-03 13:13:03 [INFO]   Max Pairs     : All


In [4]:
"""Cell 4 -- Sanity Check: Label Mapping & Schema Validation.

Guarantees that 21 classes are strictly 1-indexed (idx 0=__background__, idx 5=coin).
Prevents off-by-one label shift.
"""
sanity_check_label_mapping(_IDX_TO_NAME, logger=log)


2026-09-03 13:13:03 [INFO] [Sanity Check] Verifying class mapping contract...
2026-09-03 13:13:03 [INFO]   [PASS] 21 classes validated: idx 0='__background__', idx 1='apple', idx 5='coin', idx 20='tomato'.
2026-09-03 13:13:03 [INFO]   [PASS] No off-by-one label shift detected.


True

In [5]:
"""Cell 5 -- Load food_info.xls and density.xls."""
from src.calorie_estimation import parse_food_info
from src.faster_rcnn.eval_pipeline import load_ground_truth

FOOD_INFO_XLS = PROJECT_ROOT / "data" / "raw" / "ECUSTFD" / "paper" / "food_info.xls"
DENSITY_XLS = PROJECT_ROOT / "data" / "raw" / "ECUSTFD" / "density.xls"

food_info = parse_food_info(FOOD_INFO_XLS)
gt_by_class = load_ground_truth(DENSITY_XLS)

# Fix potential spelling alias (fired_dough_twist vs fried_dough_twist)
gt_by_class = fix_ground_truth_aliasing(gt_by_class, logger=log)

log.info("Loaded food_info: %d classes | GT density: %d classes", len(food_info), len(gt_by_class))


2026-09-03 13:13:03 [INFO] Loaded food_info: 21 classes | GT density: 20 classes


In [6]:
"""Cell 6 -- Load test split and view pairs."""
test_split_file = IMAGESETS_DIR / f"{SPLIT}.txt"
test_stems = load_split(test_split_file)
log.info("Loaded %s.txt: %d stems", SPLIT, len(test_stems))

test_paths = resolve_image_paths(test_stems, IMAGES_DIR)
log.info("Resolved image files: %d / %d stems", len(test_paths), len(test_stems))

test_groups = group_top_side(test_paths)
test_pairs = make_pairs(test_groups)
log.info("Paired views: %d (top, side) pairs resolved.", len(test_pairs))


2026-09-03 13:13:03 [INFO] Loaded test_final.txt: 928 stems
2026-09-03 13:13:03 [INFO] Resolved image files: 928 / 928 stems
2026-09-03 13:13:03 [INFO] Paired views: 6398 (top, side) pairs resolved.


In [7]:
"""Cell 7 -- Load Faster R-CNN checkpoint."""
log.info("Loading model weights from: %s", MODEL_WEIGHTS)
model = load_model(MODEL_WEIGHTS, device=DEVICE)
log.info("Model loaded successfully on device=%s.", DEVICE)


2026-09-03 13:13:03 [INFO] Loading model weights from: E:\AI_Research\dlt8\models\faster_rcnn_new_best.pt
2026-09-03 13:13:05 [INFO] Loaded Faster R-CNN from E:\AI_Research\dlt8\models\faster_rcnn_new_best.pt (device=cuda:0)
2026-09-03 13:13:05 [INFO] Model loaded successfully on device=0.


In [8]:
"""Cell 8 -- Multi-sample Smoke Test on representative images."""
sample_images = [
    IMAGES_DIR / "apple015T(1).JPG",
    IMAGES_DIR / "bread001T(1).JPG",
    IMAGES_DIR / "lemon001T(1).JPG",
]

def _smoke_predict(p, c):
    return predict_one(model, p, conf=c, iou_threshold=IOU_THRESH, device=DEVICE)
    
run_smoke_test(_smoke_predict, sample_images, conf=0.25, logger=log)


2026-09-03 13:13:05 [INFO] [Smoke Test] Testing detector + mask backend on 3 sample images...
2026-09-03 13:13:08 [INFO] predict_one(apple015T(1).JPG, conf=0.25): 2 dets, classes={'apple': 1, 'coin': 1}
2026-09-03 13:13:08 [INFO]   Sample [1/3] apple015T(1).JPG: 2 detections, classes={'apple': 1, 'coin': 1}
2026-09-03 13:13:09 [INFO] predict_one(bread001T(1).JPG, conf=0.25): 2 dets, classes={'bread': 1, 'coin': 1}
2026-09-03 13:13:09 [INFO]   Sample [2/3] bread001T(1).JPG: 2 detections, classes={'bread': 1, 'coin': 1}
2026-09-03 13:13:10 [INFO] predict_one(lemon001T(1).JPG, conf=0.25): 2 dets, classes={'coin': 1, 'lemon': 1}
2026-09-03 13:13:10 [INFO]   Sample [3/3] lemon001T(1).JPG: 2 detections, classes={'coin': 1, 'lemon': 1}
2026-09-03 13:13:10 [INFO] [Smoke Test] PASS -- All smoke test samples processed without crash.


True

In [9]:
"""Cell 9 -- Per-image End-to-End Speed Timer (Memory auto-managed by refcount cache)."""
import atexit
import time
import src.faster_rcnn.inference as frcnn_inf

_E2E_TIMES = []
_E2E_NDETS = []
_E2E_NMASK = []

# Global counters for inference timing tracking
_INFERENCE_COUNTER = 0
_CURRENT_CONF = 0.8

_orig_predict_one = frcnn_inf.predict_one

def _timed_predict_one(model_obj, image_path, conf=0.8, iou_threshold=0.3, device=None):
    """Wrapper to track inference timing. Memory managed by reference-counted cache in eval_pipeline."""
    global _INFERENCE_COUNTER, _CURRENT_CONF

    # Reset counter whenever conf threshold changes (new sweep/config)
    if conf != _CURRENT_CONF:
        _CURRENT_CONF = conf
        _INFERENCE_COUNTER = 0

    # Timing measurement
    t0 = time.perf_counter()
    dets = _orig_predict_one(model_obj, image_path, conf=conf, iou_threshold=iou_threshold, device=device)
    elapsed = time.perf_counter() - t0
    _E2E_TIMES.append(elapsed)
    _E2E_NDETS.append(len(dets))
    _E2E_NMASK.append(sum(1 for d in dets if d.get("class_name") != "coin" and d.get("mask") is not None))

    _INFERENCE_COUNTER += 1

    return dets

frcnn_inf.predict_one = _timed_predict_one

def _restore_predict_one():
    frcnn_inf.predict_one = _orig_predict_one

atexit.register(_restore_predict_one)
log.info("[Timer] Installed per-image timer wrapper for Faster R-CNN + GrabCut (memory auto-managed by refcount cache).")


2026-09-03 13:13:11 [INFO] [Timer] Installed per-image timer wrapper for Faster R-CNN + GrabCut (memory auto-managed by refcount cache).


In [10]:
"""Cell 10 -- Relaxed coin-gate patch to ensure no sample drop."""
import src.faster_rcnn.eval_pipeline as ep

def _relaxed_compute_scale(dets, fallback=0.1080):
    coins = [d for d in dets if d.get("class_name") == "coin"]
    if not coins:
        return fallback, None
    best_coin = max(coins, key=lambda d: d.get("conf", 0.0))
    bbox = best_coin["bbox"]
    w = abs(bbox[2] - bbox[0])
    h = abs(bbox[3] - bbox[1])
    diam = max(w, h)
    if diam < 5:
        return fallback, None
    scale = 2.5 / diam  # 1 Yuan coin = 2.5 cm
    return scale, best_coin

log.info("[Patch] Relaxed coin-gate active.")


2026-09-03 13:13:11 [INFO] [Patch] Relaxed coin-gate active.


In [11]:
"""Cell 11 -- Run Full E2E Faster R-CNN + GrabCut Pipeline."""
log.info("=" * 70)
log.info("Running E2E Faster R-CNN + GrabCut pipeline...")
log.info("  variant=%s, split=%s, conf=%.2f, apply_beta=True", MODEL_VARIANT, SPLIT, CONF)
log.info("=" * 70)

artifacts = ep._run_one_config(
    model_weights=MODEL_WEIGHTS,
    model_variant=MODEL_VARIANT,
    split=SPLIT,
    images_dir=IMAGES_DIR,
    imagesets_dir=IMAGESETS_DIR,
    conf_threshold=CONF,
    food_info=food_info,
    gt_by_class=gt_by_class,
    out_dir=RUN_DIR,
    apply_beta=True,
    device=DEVICE,
    mask_backend="grabcut",
    sam_predictor=None,
)

log.info("Pipeline run finished successfully. Artifacts: %s", artifacts)


2026-09-03 13:13:11 [INFO] ======================================================================
2026-09-03 13:13:11 [INFO] Running E2E Faster R-CNN + GrabCut pipeline...
2026-09-03 13:13:11 [INFO]   variant=new, split=test_final, conf=0.10, apply_beta=True
2026-09-03 13:13:11 [INFO] ======================================================================
2026-09-03 13:13:11 [INFO] === config: variant=new, split=test_final, conf=0.10, apply_beta=True, mask_backend=grabcut ===
2026-09-03 13:13:11 [INFO] Loading Faster R-CNN model: E:\AI_Research\dlt8\models\faster_rcnn_new_best.pt
2026-09-03 13:13:11 [INFO] Loaded Faster R-CNN from E:\AI_Research\dlt8\models\faster_rcnn_new_best.pt (device=cuda:0)
2026-09-03 13:13:11 [INFO] Model loaded on device=cuda:0
2026-09-03 13:13:11 [INFO] Beta calibration requested: fitting on train split (paper 50/50)
2026-09-03 13:13:11 [INFO]   Train split: 1169 stems, 6772 pairs
2026-09-03 13:13:13 [INFO] predict_one(apple001T(1).JPG, conf=0.01): 2 dets, clas

In [12]:
"""Cell 12 -- Render Per-class & Overall Markdown Report."""
import json
report = json.loads(Path(JSON_PATH).read_text(encoding="utf-8"))

df_report, report_data = format_and_print_report(
    JSON_PATH,
    title=f"Per-class ME_vol / ME_mass (Faster R-CNN + GrabCut, conf={CONF:.2f})",
)



=== Per-class ME_vol / ME_mass (Faster R-CNN + GrabCut, conf=0.10) ===
| Class             |   n |   ME_vol (%) |   |ME_vol| (%) |   ME_mass (%) |
|:------------------|----:|-------------:|---------------:|--------------:|
| peach             | 376 |        -5.25 |           9.74 |        -14.90 |
| litchi            | 128 |        -3.29 |          10.92 |         -3.26 |
| plum              | 580 |        18.58 |          19.05 |         12.25 |
| lemon             | 479 |         8.36 |          19.39 |          4.82 |
| sachima           | 661 |       -11.66 |          20.40 |        -12.59 |
| apple             | 920 |       -11.37 |          21.46 |        -12.59 |
| doughnut          | 452 |       -20.06 |          21.66 |        -23.53 |
| bread             |  81 |       -16.29 |          21.89 |        -34.70 |
| qiwi              | 231 |        -6.85 |          22.66 |         -6.65 |
| bun               |  95 |        13.05 |          24.07 |         23.97 |
| banana        

In [13]:
"""Cell 13 -- Beta Calibration Sanity Check."""
this_betas = report.get("betas", {})
log.info("Beta calibrated for %d classes.", len(this_betas))
if this_betas:
    vals = list(this_betas.values())
    log.info("  range  : [%.4f, %.4f]", min(vals), max(vals))
    log.info("  median : %.4f", float(np.median(vals)))


2026-09-03 14:29:39 [INFO] Beta calibrated for 19 classes.
2026-09-03 14:29:39 [INFO]   range  : [0.4398, 1.4787]
2026-09-03 14:29:39 [INFO]   median : 1.1553


In [14]:
"""Cell 14 -- Compute & Print Per-image Latency Report."""
cfg = {
    "model_variant": MODEL_VARIANT,
    "mask_backend": MASK_BACKEND,
    "conf_threshold": CONF,
    "iou_threshold": IOU_THRESH,
    "grabcut_iters": GRABCUT_ITERS,
    "device": DEVICE,
}

speed_stats = compute_speed_report(
    _E2E_TIMES, _E2E_NDETS, _E2E_NMASK,
    config_dict=cfg,
    run_dir=RUN_DIR,
    backend_title="Faster R-CNN + GrabCut",
    logger=log,
)


2026-09-03 14:29:39 [INFO] [timer] Speed report saved -> E:\AI_Research\dlt8\outputs\predictions\06a_faster_rcnn_eval_20260903-131303\speed_per_image.json

  PER-IMAGE END-TO-END SPEED (Faster R-CNN + GrabCut)
  n_images         : 2088
  total wall time  : 4583.41s (76.39 min)
  throughput       : 0.456 images/sec
  Per-image latency:
    mean   : 2195.1 ms  (2.1951s)
    median : 2102.3 ms  (2.1023s)
    p90    : 2704.0 ms  (2.7040s)
    p95    : 3667.0 ms  (3.6670s)
    p99    : 4756.2 ms  (4.7562s)
    min/max: 91.1 ms / 16372.1 ms
    stdev  : 770.6 ms
  avg #dets / #masks per img: 2.07 / 1.05


In [15]:
"""Cell 15 -- Write summary.txt and Manifest."""
summary_lines = [
    "=" * 70,
    "06a -- FASTER R-CNN + GRABCUT E2E EVALUATION -- SUMMARY",
    "=" * 70,
    f"Run timestamp      : {RUN_TS}",
    f"Model variant      : {MODEL_VARIANT}",
    f"Model weights      : {MODEL_WEIGHTS}",
    f"Split              : {SPLIT} ({len(test_stems)} stems)",
    f"Confidence         : {CONF}",
    f"NMS IoU threshold  : {IOU_THRESH}",
    f"GrabCut iterations : {GRABCUT_ITERS}",
    f"Device             : {DEVICE}",
    "",
    "Overall metrics:",
]
for k, v in report.get("overall", {}).items():
    val_str = f"{v:.4f}" if isinstance(v, float) else str(v)
    summary_lines.append(f"  {k:20s} = {val_str}")

summary_lines.append("")
summary_lines.append("Files written:")
summary_lines.append(f"  CSV    : {CSV_PATH}")
summary_lines.append(f"  JSON   : {JSON_PATH}")
summary_lines.append(f"  Betas  : {BETA_JSON}")
summary_lines.append(f"  Log    : {LOG_PATH}")
summary_lines.append(f"  Speed  : {RUN_DIR / 'speed_per_image.json'}")

summary_path = RUN_DIR / "summary.txt"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
log.info("Summary written -> %s", summary_path)

print("\n" + "=" * 70)
print("  ALL OUTPUT FILES (06a):")
print("=" * 70)
print(f"  Log            : {LOG_PATH}")
print(f"  Run dir        : {RUN_DIR}")
print(f"  Predictions CSV: {CSV_PATH}")
print(f"  Report JSON    : {JSON_PATH}")
print(f"  Betas JSON     : {BETA_JSON}")
print(f"  Speed JSON     : {RUN_DIR / 'speed_per_image.json'}")
print(f"  Summary txt    : {summary_path}")
print("=" * 70)
print("\n>>> Notebook 06a is complete.")


2026-09-03 14:29:39 [INFO] Summary written -> E:\AI_Research\dlt8\outputs\predictions\06a_faster_rcnn_eval_20260903-131303\summary.txt

  ALL OUTPUT FILES (06a):
  Log            : E:\AI_Research\dlt8\outputs\logs\06a_faster_rcnn_eval_20260903-131303.log
  Run dir        : E:\AI_Research\dlt8\outputs\predictions\06a_faster_rcnn_eval_20260903-131303
  Predictions CSV: E:\AI_Research\dlt8\outputs\predictions\06a_faster_rcnn_eval_20260903-131303\samples_test_final_conf10_beta.csv
  Report JSON    : E:\AI_Research\dlt8\outputs\predictions\06a_faster_rcnn_eval_20260903-131303\report_test_final_conf10_beta.json
  Betas JSON     : E:\AI_Research\dlt8\outputs\predictions\06a_faster_rcnn_eval_20260903-131303\betas_train_conf10.json
  Speed JSON     : E:\AI_Research\dlt8\outputs\predictions\06a_faster_rcnn_eval_20260903-131303\speed_per_image.json
  Summary txt    : E:\AI_Research\dlt8\outputs\predictions\06a_faster_rcnn_eval_20260903-131303\summary.txt

>>> Notebook 06a is complete.
